# Part I. ETL Pipeline for Pre-Processing the Files

## PLEASE RUN THE FOLLOWING CODE FOR PRE-PROCESSING THE FILES

#### Import Python packages 

In [ ]:
# Import Python packages 
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import json
import csv

#### Creating list of filepaths to process original event csv data files

In [ ]:
# checking your current working directory
print(os.getcwd())

# Get your current folder and subfolder event data
filepath = os.getcwd() + '/event_data'

# Create a for loop to create a list of files and collect each filepath
for root, dirs, files in os.walk(filepath):
    
# join the file path and roots with the subdirectories using glob
    file_path_list = glob.glob(os.path.join(root,'*'))
    #print(file_path_list)

/workspace/home


#### Processing the files to create the data file csv that will be used for Apache Casssandra tables

In [ ]:
# initiating an empty list of rows that will be generated from each file
full_data_rows_list = [] 
    
# for every filepath in the file path list 
for f in file_path_list:

# reading csv file 
    with open(f, 'r', encoding = 'utf8', newline='') as csvfile: 
        # creating a csv reader object 
        csvreader = csv.reader(csvfile) 
        next(csvreader)
        
 # extracting each data row one by one and append it        
        for line in csvreader:
            #print(line)
            full_data_rows_list.append(line) 
            
# uncomment the code below if you would like to get total number of rows 
#print(len(full_data_rows_list))
# uncomment the code below if you would like to check to see what the list of event data rows will look like
#print(full_data_rows_list)

# creating a smaller event data csv file called event_datafile_full csv that will be used to insert data into the \
# Apache Cassandra tables
csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

with open('event_datafile_new.csv', 'w', encoding = 'utf8', newline='') as f:
    writer = csv.writer(f, dialect='myDialect')
    writer.writerow(['artist','firstName','gender','itemInSession','lastName','length',\
                'level','location','sessionId','song','userId'])
    for row in full_data_rows_list:
        if (row[0] == ''):
            continue
        writer.writerow((row[0], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[12], row[13], row[16]))


In [ ]:
# check the number of rows in your csv file
with open('event_datafile_new.csv', 'r', encoding = 'utf8') as f:
    print(sum(1 for line in f))

6821


# Part II. Complete the Apache Cassandra coding portion of your project. 

## Now you are ready to work with the CSV file titled <font color=red>event_datafile_new.csv</font>, located within the Workspace directory.  The event_datafile_new.csv contains the following columns: 
- artist 
- firstName of user
- gender of user
- item number in session
- last name of user
- length of the song
- level (paid or free song)
- location of the user
- sessionId
- song title
- userId

The image below is a screenshot of what the denormalized data should appear like in the <font color=red>**event_datafile_new.csv**</font> after the code above is run:<br>

<img src="images/image_event_datafile_new.jpg">

## Begin writing your Apache Cassandra code in the cells below

#### Creating a Cluster

In [ ]:
# This should make a connection to a Cassandra instance your local machine 
# (127.0.0.1)

from cassandra.cluster import Cluster
cluster = Cluster()

# To establish connection and begin executing queries, need a session
session = cluster.connect()

#### Create Keyspace

In [ ]:
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS udacity 
    WITH REPLICATION = 
    { 'class' : 'SimpleStrategy', 'replication_factor' : 1 }"""
)

except Exception as e:
    print(e)
    

#### Set Keyspace

In [ ]:
try:
    session.set_keyspace('udacity')
except Exception as e:
    print(e)

### Now we need to create tables to run the following queries. Remember, with Apache Cassandra you model the database tables on the queries you want to run.

In [ ]:
query = """
CREATE TABLE IF NOT EXISTS song_library (
    sessionId int,
    itemInSession int,
    artist text,
    song text,
    length float,
    PRIMARY KEY (sessionId, itemInSession)
)
"""

session.execute(query)

In [ ]:
file = 'event_datafile_new.csv'

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)

    for line in csvreader:

        query = """
        INSERT INTO song_library (
            sessionId,
            itemInSession,
            artist,
            song,
            length
        )
        VALUES (%s, %s, %s, %s, %s)
        """

        session.execute(query, (
            int(line[8]),
            int(line[3]),
            line[0],
            line[9],
            float(line[5])
        ))

In [ ]:
query = """
CREATE TABLE IF NOT EXISTS user_song_library (
    userId int,
    sessionId int,
    itemInSession int,
    artist text,
    song text,
    firstName text,
    lastName text,
    PRIMARY KEY ((userId), sessionId, itemInSession)
)
"""

session.execute(query)

In [ ]:
file = 'event_datafile_new.csv'

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)

    for line in csvreader:

        query = """
        INSERT INTO user_song_library (
            userId,
            sessionId,
            itemInSession,
            artist,
            song,
            firstName,
            lastName
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """

        session.execute(query, (
            int(line[10]),
            int(line[8]),
            int(line[3]),
            line[0],
            line[9],
            line[1],
            line[4]
        ))

In [ ]:
query = """
CREATE TABLE IF NOT EXISTS song_user_library (
    song text,
    userId int,
    firstName text,
    lastName text,
    PRIMARY KEY (song, userId)
)
"""

session.execute(query)

In [ ]:
file = 'event_datafile_new.csv'

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)

    for line in csvreader:

        query = """
        INSERT INTO song_user_library (
            song,
            userId,
            firstName,
            lastName
        )
        VALUES (%s, %s, %s, %s)
        """

        session.execute(query, (
            line[9],
            int(line[10]),
            line[1],
            line[4]
        ))

In [ ]:
rows = session.execute("""
SELECT * FROM user_song_library LIMIT 5
""")

df = pd.DataFrame(list(rows))

df

,userid,sessionid,iteminsession,artist,firstname,lastname,song
0,23,177,2,Dwight Yoakam,Morris,Gilmore,You're The One
1,23,351,0,Foals,Morris,Gilmore,Blue Blood
2,23,351,1,'N Sync/Phil Collins,Morris,Gilmore,Trashin' The Camp (Phil And 'N Sync Version)
3,23,841,0,Eminem,Morris,Gilmore,Just Lose It
4,53,52,2,Mynt,Celeste,Williams,Playa Haters


## Create queries to ask the following three questions of the data

### 1. Give me the artist, song title and song's length in the music app history that was heard during  sessionId = 338, and itemInSession  = 4


### 2. Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name) for userid = 10, sessionid = 182
    

### 3. Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'




In [ ]:
##Query 1:This table was designed to retrieve artist name,song title and song length based on sessionIdand itemInSession.

In [ ]:
import pandas as pd

rows = session.execute("""
SELECT artist, song, length
FROM song_library
WHERE sessionId = 338
AND itemInSession = 4
""")

df = pd.DataFrame(list(rows))
df

,artist,song,length
0,Faithless,Music Matters (Mark Knight Dub),495.307312


In [ ]:
query = """
SELECT artist, song, firstName, lastName
FROM user_song_library
WHERE userId = 10
AND sessionId = 182
"""

rows = session.execute(query)

df = pd.DataFrame(list(rows))

df

,artist,song,firstname,lastname
0,Down To The Bone,Keep On Keepin' On,Sylvie,Cruz
1,Three Drives,Greece 2000,Sylvie,Cruz
2,Sebastien Tellier,Kilometer,Sylvie,Cruz
3,Lonnie Gordon,Catch You Baby (Steve Pitron & Max Sanna Radio...,Sylvie,Cruz


In [ ]:
query = """
SELECT firstName, lastName
FROM song_user_library
WHERE song = 'All Hands Against His Own'
"""
rows = session.execute(query)

df = pd.DataFrame(list(rows))
df

,firstname,lastname
0,Jacqueline,Lynch
1,Tegan,Levine
2,Sara,Johnson


#### Do a SELECT to verify that the data have been inserted into each table

### COPY AND REPEAT THE ABOVE THREE CELLS FOR EACH OF THE THREE QUESTIONS

In [ ]:
rows = session.execute("""
SELECT artist, song, length
FROM song_library
WHERE sessionId = 338
AND itemInSession = 4
""")

df = pd.DataFrame(list(rows))

display(df)

,artist,song,length
0,Faithless,Music Matters (Mark Knight Dub),495.307312


In [ ]:
## Query 2: Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name)\
## for userid = 10, sessionid = 182
# QUERY 2

rows = session.execute("""
SELECT artist, song, firstName, lastName
FROM user_song_library
WHERE userId = 10
AND sessionId = 182
""")

df = pd.DataFrame(list(rows))

display(df)

,artist,song,firstname,lastname
0,Down To The Bone,Keep On Keepin' On,Sylvie,Cruz
1,Three Drives,Greece 2000,Sylvie,Cruz
2,Sebastien Tellier,Kilometer,Sylvie,Cruz
3,Lonnie Gordon,Catch You Baby (Steve Pitron & Max Sanna Radio...,Sylvie,Cruz


In [ ]:


# QUERY 3

rows = session.execute("""
SELECT firstName, lastName
FROM song_user_library
WHERE song = 'All Hands Against His Own'
""")


df = pd.DataFrame(list(rows))

display(df)
                    

,firstname,lastname
0,Jacqueline,Lynch
1,Tegan,Levine
2,Sara,Johnson


### Drop the tables before closing out the sessions

In [ ]:
## Drop the table before closing out the sessions

In [ ]:
session.execute("DROP TABLE IF EXISTS song_library")

In [ ]:
session.execute("DROP TABLE IF EXISTS user_song_library")

In [ ]:
session.execute("DROP TABLE IF EXISTS song_user_library")

### Close the session and cluster connection¶

In [ ]:
session.shutdown()
cluster.shutdown()